In [2]:
from dotenv import load_dotenv
load_dotenv()

True

In [3]:
from pathlib import Path
from urllib.request import urlretrieve

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_core.vectorstores import InMemoryVectorStore
from langchain.tools import tool
from langchain.agents import create_agent
from langchain.messages import HumanMessage

In [4]:
# Download a public  employee handbook PDF

HANDBOOK_URL = "https://www.grayson.edu/employment/employee-handbook-2023.2024-rev07.26.20231.pdf"
LOCAL_PDF = Path("../assets/pdf/EMPLOYEE-HANDBOOK.pdf").resolve()

LOCAL_PDF.parent.mkdir(parents=True, exist_ok=True)

In [ ]:
# Load the PDF

# PyPDFLoader reads a PDF file from disk and converts each page into a LangChain document object
# Each document has two parts: .page_content and .metadata
loader = PyPDFLoader(str(LOCAL_PDF.resolve()))
docs = loader.load()

print(f"Loaded {len(docs)} pages from handbook")
print(docs[0].metadata)

Loaded 48 pages from handbook
{'producer': 'Microsoft® Word for Microsoft 365', 'creator': 'Microsoft® Word for Microsoft 365', 'creationdate': '2024-02-01T13:42:37-05:00', 'author': 'gretac', 'moddate': '2024-02-01T13:42:37-05:00', 'source': '/Users/angelo/projects/genai-portfolio/projects/01-langchain/assets/pdf/EMPLOYEE-HANDBOOK.pdf', 'total_pages': 48, 'page': 0, 'page_label': '1'}


In [6]:
# split the document into chunks
"""
Best practice
chunk_size large enough to preserve policy meaning
chunk_overlap to avoid cutting rules mid thought
"""

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    add_start_index=True,
)

splits = text_splitter.split_documents(docs)
print(f"Created {len(splits)} chunks")

Created 119 chunks


In [8]:
# Build embeddings + vector store
# For learning, InMemoryVectorStore is perfect.
# In production, you'd likely use FAISS, Chroma, pgvector, Pinecone, etc.

embeddings = OpenAIEmbeddings(model="text-embedding-3-large")
vector_store = InMemoryVectorStore(embeddings)
_ = vector_store.add_documents(documents=splits)

AuthenticationError: Error code: 401 - {'error': {'message': 'Incorrect API key provided: your_rea******here. You can find your API key at https://platform.openai.com/account/api-keys.', 'type': 'invalid_request_error', 'param': None, 'code': 'invalid_api_key'}}